#Battery Storage Cost Savings Analysis for California utilities

**Research Question**: How much money can California utilities save by using large batteries to shift electricity from cheap midday hours to expensive evening peaks?

**Hypothesis**: By soaking up cheap solar power mid-day and discharging during evening spikes, utilities can reduce energy procurement costs without changing total consumption.

**Data Source**: CAISO OASIS  (https://oasis.caiso.com/mrioasis/logon.do?reason=application.baseAction.noSession#)
  - Demand side: System Load and Resources Schedules Data
  - Supply side: Public Bids Data

---


**Context / Market Mechanism:**

1. How Prices are Set: The Merit Order
*   In the California ISO (CAISO) market, electricity prices are determined by a "reverse auction." Generators submit bids (the supply stack) starting from the cheapest (solar/wind) to the most expensive (natural gas peakers).

*   The Marginal Clearing Price is set by the very last megawatt needed to meet demand. Crucially, every megawatt bought in that hour is paid that same clearing price. When the demand is high and every generator supplies power, the clearing price is set by those with the highest marginal cost (such as natural gas peaker plants).

2. The Battery's Role:
*   When Charging, the battery acts as "New Demand," potentially pushing the market into a more expensive "step" on the supply stack.
*   When Discharging: The battery acts as "Negative Demand," potentially allowing the grid to turn off an expensive gas plant, lowering the price for all consumers during that hour **(Price Supression Effect)**.

In CAISO, **the California Public Utilities Commission (CPUC) has integrated large-scale storage into its Integrated Resource Plan (IRP), banking on the idea that storage will reduce system-wide costs** ([CPUC, 2023](https://www.cpuc.ca.gov/-/media/cpuc-website/divisions/energy-division/documents/integrated-resource-plan-and-long-term-procurement-plan-irp-ltpp/2023-irp-cycle-events-and-materials/2022-2023psp_decision_2pager_final.pdf)). Furthermore, a flexible industrial facility (like a data center) can act exactly like a large battery system, providing demand-side flexibility. The California Energy Commission (CEC) recently established a goal of 7,000 MW of flexible load by 2030 to help stabilize the grid ([CEC, 2024](https://www.energy.ca.gov/programs-and-topics/topics/load-flexibility)).

**Therefore, this project aims to validate (or disprove) this hypothesis with real-world data and answer the question of "to what extent."**



*   For instance, how much money can the grid system save with battery?
*   Does the 100th battery save as much money as the 1st battery? What's the saturation point where adding more batteries stops being profitable?

---



### II. Methodology

To answer my research question, I built an **energy & battery dispatch simulation engine** that mimics how the CAISO market clears in real-time.

For each hour, we applied a MW adjustment based on our battery (or flexible industrial load) behavior:
*   During Solar Hours (9 AM – 3 PM): We added MWs to the load (Charging/Pre-cooling).
*   During Peak Hours (5 PM – 9 PM): We subtracted MWs from the load (Discharging/Curtailing).

This gave us a new value: the Net Load for each hour. For each hour, we find the electricity price corresponding to the new net load.

Unlike simple arbitrage models that assume prices are "fixed," the method accounts for Price Elasticity - the fact that adding load makes energy more expensive, and removing load makes it cheaper.

## (1). Data Reconstruction and Load Profile Exploration:

Begin by merging two massive datasets from the CAISO OASIS portal for November 30, 2025:

*   The Demand Side: Hourly "System Load" data representing the total electricity consumption of California.
*   The Supply Side: "Public Bid" data, which contains thousands of individual bids from power plants. We reconstructed these into a Supply Stack (Merit Order) for every hour of the day, showing exactly how much power is available at every price point.

For analytical purpose, we also assume the below **battery parameters**:
  - Capacity: 1000 MWh (roughly a large utility-scale BESS)
  - Power rating: 250 MW (4-hour battery)
  - Round-trip efficiency: 90%
  - Charge window: 09:00–15:00 (cheap solar hours)
  - Discharge window: 17:00–21:00 (evening peak)

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ===========================================================================
# CONFIG
# ===========================================================================
LOAD_FILE   = "20251130_20251201_SLD_FCST_DAM_20260302_08_09_23_v1.csv"
BID_FILE    = "20251130_20251130_PUB_BID_DAM_v3.csv"
TARGET_DATE = "2025-11-30"

BATTERY_CAPACITY_MWH = 1000
BATTERY_POWER_MW     = 250
ROUND_TRIP_EFF       = 0.90
CHARGE_HOURS         = list(range(9, 16))
DISCHARGE_HOURS      = list(range(17, 22))

# ===========================================================================
# LOAD RAW FILES
# ===========================================================================
load_raw = pd.read_csv(LOAD_FILE)
bid_raw  = pd.read_csv(BID_FILE)

# ===========================================================================
# SECTION 1 — GROUPBY
# Find peak hour by grouping load data by hour, then aggregating with sum.
# CA ISO-TAC is the system total but there may be multiple rows per hour
# (sub-areas, intervals). Groupby + sum is the safe, correct way to get
# total system load per hour rather than just grabbing one row.
# ===========================================================================
print("=" * 65)
print("SECTION 1: Hourly Load Profile")
print("=" * 65)

# Filter to CA ISO-TAC and target date first
load_isotac = load_raw[
    (load_raw["TAC_AREA_NAME"] == "CA ISO-TAC") &
    (load_raw["OPR_DT"]        == TARGET_DATE)
].copy()
load_isotac["OPR_HR"] = load_isotac["OPR_HR"].astype(int)

# --- GROUPBY: sum MW across any duplicate rows for the same hour ---
# Result: one row per hour with total system load
hourly_load = (
    load_isotac
    .groupby("OPR_HR", as_index=False)   # group by hour-ending
    .agg(TOTAL_MW=("MW", "sum"))          # named aggregation
)
hourly_load = hourly_load.sort_values("OPR_HR").reset_index(drop=True)


# Auto-detect peak hour from the grouped result
peak_row     = hourly_load.loc[hourly_load["TOTAL_MW"].idxmax()]
PEAK_HE      = int(peak_row["OPR_HR"])
peak_load_mw = float(peak_row["TOTAL_MW"])

print(f"  Date         : {TARGET_DATE}")
print(f"  Peak Hour    : HE {PEAK_HE}  (detected)")
print(f"  Peak Load    : {peak_load_mw:,.2f} MW")

print(f"\n  Full daily load profile:")
for _, row in hourly_load.iterrows():
    he  = int(row["OPR_HR"])
    mw  = float(row["TOTAL_MW"])

    # Generate a bar chart
    bar = "█" * int(mw / 500)
    tag = "  ◀ PEAK" if he == PEAK_HE else \
          "  [charge]" if he in CHARGE_HOURS else \
          "  [discharge]" if he in DISCHARGE_HOURS else ""

    print(f"    HE {he:>2}  {mw:>9,.2f} MW  {bar}{tag}")

SECTION 1: Hourly Load Profile
  Date         : 2025-11-30
  Peak Hour    : HE 19  (detected)
  Peak Load    : 24,577.31 MW

  Full daily load profile:
    HE  1  20,679.10 MW  █████████████████████████████████████████
    HE  2  19,929.35 MW  ███████████████████████████████████████
    HE  3  19,284.59 MW  ██████████████████████████████████████
    HE  4  18,947.45 MW  █████████████████████████████████████
    HE  5  18,905.50 MW  █████████████████████████████████████
    HE  6  19,269.45 MW  ██████████████████████████████████████
    HE  7  19,870.29 MW  ███████████████████████████████████████
    HE  8  20,070.81 MW  ████████████████████████████████████████
    HE  9  19,926.29 MW  ███████████████████████████████████████  [charge]
    HE 10  19,101.01 MW  ██████████████████████████████████████  [charge]
    HE 11  18,369.73 MW  ████████████████████████████████████  [charge]
    HE 12  17,593.18 MW  ███████████████████████████████████  [charge]
    HE 13  17,234.85 MW  ██████████████

Analysis:
Notice that in just 5 hours - HE 13 to HE 18 - the grid has to find around 7,400 MW (= 24,600 - 17,200 MW) of new generation. By using batteries to "shave" that peak, we are directly reducing the stress on the California grid and improving energy security / reliability.

## (2). Price Re-Discovery and Battery Dispatch Simulation

This is the core of our method. We construct a **supply bid stack** using public bid data. For every hour, the simulation "looks up" the corresponding bid price with the new Net Load in our reconstructed supply stack.

Eg. If the original load was 24,577 MW and our battery discharged 1,000 MW, the simulation finds the clearing price for 23,577 MW.

The difference between the Original Clearing Price and the New Price represents the "**Price Suppression**" value created by the flexibility.

In [3]:

# ===========================================================================
# SECTION 2 — RESHAPE (pivot_table)
# The bid file has one row per bid-curve point. We want a supply stack:
# for each price level, how many total MW are offered system-wide?
# pivot_table lets us reshape from long (one row per generator-point)
# to a summary (one row per price bucket, total MW across all generators).
# ===========================================================================
print(f"\n{'=' * 65}")
print("SECTION 2: Build Supply Stack via pivot_table (reshape)")
print("=" * 65)

# Filter to generator price bids for the peak hour
peak_hour_start = f"{TARGET_DATE}T{(PEAK_HE-1):02d}:00:00"

bids_raw = bid_raw[
    (bid_raw["RESOURCE_TYPE"]     == "GENERATOR") &
    (bid_raw["SCH_BID_CURVETYPE"] == "BIDPRICE")  &
    (bid_raw["STARTTIME"]         == peak_hour_start) &
    (bid_raw["SCH_BID_XAXISDATA"].notna()) &
    (bid_raw["SCH_BID_Y1AXISDATA"].notna())
].copy()

bids_raw["MW"]    = pd.to_numeric(bids_raw["SCH_BID_XAXISDATA"],  errors="coerce")
bids_raw["PRICE"] = pd.to_numeric(bids_raw["SCH_BID_Y1AXISDATA"], errors="coerce")
bids_raw = bids_raw.dropna(subset=["MW", "PRICE"])

print(f"  Raw bid-curve rows: {len(bids_raw):,}")
print(f"  Each row = one (MW, price) point on one generator's offer curve")

# --- RESHAPE with pivot_table ---
# Round price to nearest dollar to create discrete price buckets.
# pivot_table aggregates all generators offering at the same price level,
# summing their MW — turning thousands of generator-level rows into a
# compact price→MW supply table.
bids_raw["PRICE_BUCKET"] = bids_raw["PRICE"].round(2)

supply_by_price = pd.pivot_table(
    bids_raw,
    values="MW",            # what to aggregate
    index="PRICE_BUCKET",   # rows = price levels
    aggfunc="sum"           # sum all MW offered at each price
).reset_index()
supply_by_price.columns = ["PRICE", "MW_AT_PRICE"]
supply_by_price = supply_by_price.sort_values("PRICE").reset_index(drop=True)

# Cumulative sum builds the step-function supply curve
supply_by_price["CUMULATIVE_MW"] = supply_by_price["MW_AT_PRICE"].cumsum()

print(f"\n  After pivot_table: {len(supply_by_price)} price buckets (was {len(bids_raw):,} rows)")
print(f"  pivot_table(values='MW', index='PRICE_BUCKET', aggfunc='sum'):")
print(supply_by_price.head(10).to_string(index=False))
print(f"  ... total offered: {supply_by_price['CUMULATIVE_MW'].iloc[-1]:,.1f} MW")

# ===========================================================================
# SECTION 3 — MERGE
# We have two separate DataFrames:
#   hourly_load  : one row per hour, total system load MW
#   supply stacks: built per hour from bid data
# We'll build per-hour supply stacks, extract clearing prices,
# then MERGE that price table back onto the load table so every
# hour has both its load MW and its clearing price in one place.
# ===========================================================================
print(f"\n{'=' * 65}")
print("SECTION 3: Clearing Prices via per-hour stacks, then merge")
print("=" * 65)

def build_supply_stack(bid_df, date, he):
    """Build aggregated supply stack for one hour using pivot_table."""
    hour_start = f"{date}T{(he-1):02d}:00:00"
    bids = bid_df[
        (bid_df["RESOURCE_TYPE"]     == "GENERATOR") &
        (bid_df["SCH_BID_CURVETYPE"] == "BIDPRICE")  &
        (bid_df["STARTTIME"]         == hour_start)  &
        (bid_df["SCH_BID_XAXISDATA"].notna()) &
        (bid_df["SCH_BID_Y1AXISDATA"].notna())
    ].copy()
    bids["MW"]    = pd.to_numeric(bids["SCH_BID_XAXISDATA"],  errors="coerce")
    bids["PRICE"] = pd.to_numeric(bids["SCH_BID_Y1AXISDATA"], errors="coerce")
    bids = bids.dropna(subset=["MW","PRICE"])
    if bids.empty:
        return None
    bids["PRICE_BUCKET"] = bids["PRICE"].round(2)
    stk = pd.pivot_table(bids, values="MW", index="PRICE_BUCKET",
                         aggfunc="sum").reset_index()
    stk.columns = ["PRICE", "MW_AT_PRICE"]
    stk = stk.sort_values("PRICE").reset_index(drop=True)
    stk["CUMULATIVE_MW"] = stk["MW_AT_PRICE"].cumsum()
    return stk

def lookup_clearing_price(load_mw, supply_stack):
    """
    Marginal price: first price bucket where cumulative supply >= load.
    Uses boolean filter instead of searchsorted to correctly handle
    non-monotonic cumulative MW caused by negative bids in the stack.
    """
    sufficient = supply_stack[supply_stack["CUMULATIVE_MW"] >= load_mw]
    if sufficient.empty:
        return float(supply_stack["PRICE"].iloc[-1])
    return float(sufficient.iloc[0]["PRICE"])

# Build a clearing-price table: one row per hour
all_hours = sorted(set(CHARGE_HOURS + DISCHARGE_HOURS + [PEAK_HE]))
price_rows = []

for he in all_hours:
    stk = build_supply_stack(bid_raw, TARGET_DATE, he)
    if stk is None:
        continue
    load_rows = hourly_load[hourly_load["OPR_HR"] == he]
    if load_rows.empty:
        continue
    load_mw = float(load_rows["TOTAL_MW"].values[0])
    price   = lookup_clearing_price(load_mw, stk)
    price_rows.append({"OPR_HR": he, "CLEARING_PRICE": price, "STACK": stk})

prices_df = pd.DataFrame([{k: v for k, v in r.items() if k != "STACK"}
                           for r in price_rows])

print("  Clearing prices table (before merge):")
print(prices_df.to_string(index=False))

# --- MERGE: join load MW and clearing price on OPR_HR ---
# Combining two separate DataFrames on a shared key.
# Left join keeps all hours from hourly_load; matching hours get a price.
hourly_combined = pd.merge(
    hourly_load,       # left:  OPR_HR, TOTAL_MW
    prices_df,         # right: OPR_HR, CLEARING_PRICE
    on="OPR_HR",       # shared key column
    how="left"         # keep all load hours; non-matched get NaN price
)

print(f"\n  After merge(hourly_load, prices_df, on='OPR_HR', how='left'):")
print(hourly_combined[["OPR_HR","TOTAL_MW","CLEARING_PRICE"]].to_string(index=False))


SECTION 2: Build Supply Stack via pivot_table (reshape)
  Raw bid-curve rows: 776
  Each row = one (MW, price) point on one generator's offer curve

  After pivot_table: 399 price buckets (was 776 rows)
  pivot_table(values='MW', index='PRICE_BUCKET', aggfunc='sum'):
  PRICE  MW_AT_PRICE  CUMULATIVE_MW
-150.00       101.44         101.44
-149.99       -29.90          71.54
-149.00         0.00          71.54
-148.88        -6.90          64.64
-108.88        -6.80          57.84
-100.00        -0.50          57.34
 -88.88        -6.50          50.84
 -83.60         8.25          59.09
 -73.64        76.66         135.75
 -70.00       163.00         298.75
  ... total offered: 77,022.1 MW

SECTION 3: Clearing Prices via per-hour stacks, then merge
  Clearing prices table (before merge):
 OPR_HR  CLEARING_PRICE
      9           16.00
     10           31.57
     11           19.90
     12           20.93
     13           22.21
     14           21.60
     15           31.20
     17   

In [4]:


# ===========================================================================
# SECTION 4 — Battery dispatch: compute net loads for charge/discharge hours
# ===========================================================================
print(f"\n{'=' * 65}")
print("SECTION 4: Battery Dispatch")
print("=" * 65)

n_charge          = len(CHARGE_HOURS)
n_discharge       = len(DISCHARGE_HOURS)
charge_energy     = min(BATTERY_POWER_MW * n_charge, BATTERY_CAPACITY_MWH)
charge_mw         = charge_energy / n_charge
energy_stored     = charge_energy * ROUND_TRIP_EFF
discharge_mw      = min(BATTERY_POWER_MW, energy_stored / n_discharge)

print(f"  Charge   : {charge_mw:.1f} MW × {n_charge} hours = {charge_energy:.0f} MWh drawn")
print(f"  Stored   : {energy_stored:.0f} MWh after {ROUND_TRIP_EFF:.0%} round-trip efficiency")
print(f"  Discharge: {discharge_mw:.1f} MW × {n_discharge} hours = {discharge_mw*n_discharge:.0f} MWh delivered")

# Assign battery role to each hour
def battery_role(he):
    if he in DISCHARGE_HOURS: return "discharge"
    if he in CHARGE_HOURS:    return "charge"
    return "none"

def battery_mw_adj(he):
    """Negative = battery reduces net load (discharge), positive = raises it (charge)."""
    if he in DISCHARGE_HOURS: return -discharge_mw
    if he in CHARGE_HOURS:    return +charge_mw
    return 0.0

hourly_combined["ROLE"]        = hourly_combined["OPR_HR"].apply(battery_role)
hourly_combined["BATTERY_ADJ"] = hourly_combined["OPR_HR"].apply(battery_mw_adj)
hourly_combined["NET_LOAD_MW"] = hourly_combined["TOTAL_MW"] + hourly_combined["BATTERY_ADJ"]

# Compute new clearing price for each affected hour
stacks_by_he = {r["OPR_HR"]: r["STACK"] for r in price_rows}

def get_new_price(row):
    he  = row["OPR_HR"]
    stk = stacks_by_he.get(he)
    if stk is None or row["ROLE"] == "none":
        return row["CLEARING_PRICE"]
    return lookup_clearing_price(row["NET_LOAD_MW"], stk)

hourly_combined["NEW_PRICE"]    = hourly_combined.apply(get_new_price, axis=1)
hourly_combined["PRICE_DELTA"]  = hourly_combined["CLEARING_PRICE"] - hourly_combined["NEW_PRICE"]
hourly_combined["BASE_COST"]    = hourly_combined["CLEARING_PRICE"] * hourly_combined["TOTAL_MW"]
hourly_combined["NEW_COST"]     = hourly_combined["NEW_PRICE"]      * hourly_combined["NET_LOAD_MW"]
hourly_combined["HOURLY_SAVING"]= hourly_combined["BASE_COST"]      - hourly_combined["NEW_COST"]

# ===========================================================================
# SECTION 5 — GROUPBY AGAIN: summarise savings by role
# Group the results by battery role to compare charge vs discharge hours.
# ===========================================================================
print(f"\n{'=' * 65}")
print("SECTION 5: Savings Summary via groupby on Role")
print("=" * 65)

# Filter to only hours with a battery role
active_hours = hourly_combined[hourly_combined["ROLE"] != "none"].copy()

print("  Hour-by-hour results:")
cols = ["OPR_HR","ROLE","TOTAL_MW","BATTERY_ADJ","NET_LOAD_MW",
        "CLEARING_PRICE","NEW_PRICE","PRICE_DELTA","HOURLY_SAVING"]
print(active_hours[cols].to_string(index=False))

# --- GROUPBY: aggregate savings by role (charge vs discharge) ---
role_summary = (
    active_hours
    .groupby("ROLE", as_index=False)
    .agg(
        HOURS        = ("OPR_HR",       "count"),
        TOTAL_SAVING = ("HOURLY_SAVING", "sum"),
        AVG_BASE_PX  = ("CLEARING_PRICE","mean"),
        AVG_NEW_PX   = ("NEW_PRICE",     "mean"),
        AVG_PX_DELTA = ("PRICE_DELTA",   "mean"),
    )
)

print(f"\n  groupby('ROLE').agg(sum/mean) → savings by role:")
print(role_summary.to_string(index=False))

# ===========================================================================
# SECTION 6 — RESHAPE (melt): wide → long for a clean final report
# We have BASE_COST and NEW_COST as separate columns (wide format).
# melt() unpivots them into a long format: one row per (hour, scenario),
# which is easier to read as a comparison table and ready for plotting.
# ===========================================================================
print(f"\n{'=' * 65}")
print("SECTION 6: Reshape with melt (comparison table)")
print("=" * 65)

# Select the wide-format cost columns
cost_wide = active_hours[["OPR_HR", "ROLE", "BASE_COST", "NEW_COST"]].copy()

# --- MELT: unpivot BASE_COST / NEW_COST into a single 'SCENARIO' column ---
cost_long = pd.melt(
    cost_wide,
    id_vars    = ["OPR_HR", "ROLE"],       # columns to keep as-is
    value_vars = ["BASE_COST", "NEW_COST"], # columns to unpivot
    var_name   = "SCENARIO",               # new column naming the source column
    value_name = "COST_USD"                # new column holding the values
)
cost_long = cost_long.sort_values(["OPR_HR", "SCENARIO"]).reset_index(drop=True)

print(f"\n one row per (hour × scenario):")
print(cost_long.to_string(index=False))

# ===========================================================================
# SECTION 7 — Final P&L
# ===========================================================================
print(f"\n{'=' * 65}")
print("SECTION 7: Daily P&L")
print("=" * 65)

discharge_saving = role_summary.loc[role_summary["ROLE"]=="discharge","TOTAL_SAVING"].sum()
charge_cost      = role_summary.loc[role_summary["ROLE"]=="charge",   "TOTAL_SAVING"].sum()
net_benefit      = discharge_saving + charge_cost

avg_discharge_px = role_summary.loc[role_summary["ROLE"]=="discharge","AVG_BASE_PX"].values[0]
avg_charge_px    = role_summary.loc[role_summary["ROLE"]=="charge",   "AVG_BASE_PX"].values[0]
arb_spread       = avg_discharge_px - (avg_charge_px / ROUND_TRIP_EFF)
arb_revenue      = arb_spread * discharge_mw * n_discharge

print(f"  Base Cost (CLEARING_PRICE × TOTAL_MW)   : ${discharge_saving:>12,.2f}/day")
print(f"  New Cost (NEW_PRICE × NET_LOAD_MW)      : ${charge_cost:>12,.2f}/day")
print(f"  ──────────────────────────────────────────────────────")
print(f"  Net price suppression benefit           : ${net_benefit:>12,.2f}/day")
print(f"\n  Energy arbitrage:")
print(f"    Avg discharge price                : ${avg_discharge_px:>10.2f}/MWh")
print(f"    Avg charge price                   : ${avg_charge_px:>10.2f}/MWh")
print(f"    Effective spread (after losses)    : ${arb_spread:>10.2f}/MWh")
print(f"    Arbitrage revenue                  : ${arb_revenue:>10,.2f}/day")
print(f"  ──────────────────────────────────────────────────────")
print(f"  Total estimated daily benefit        : ${net_benefit + arb_revenue:>12,.2f}/day")


SECTION 4: Battery Dispatch
  Charge   : 142.9 MW × 7 hours = 1000 MWh drawn
  Stored   : 900 MWh after 90% round-trip efficiency
  Discharge: 180.0 MW × 5 hours = 900 MWh delivered

SECTION 5: Savings Summary via groupby on Role
  Hour-by-hour results:
 OPR_HR      ROLE  TOTAL_MW  BATTERY_ADJ  NET_LOAD_MW  CLEARING_PRICE  NEW_PRICE  PRICE_DELTA  HOURLY_SAVING
      9    charge  19926.29   142.857143 20069.147143           16.00      16.00         0.00   -2285.714286
     10    charge  19101.01   142.857143 19243.867143           31.57      31.57         0.00   -4510.000000
     11    charge  18369.73   142.857143 18512.587143           19.90      19.90         0.00   -2842.857143
     12    charge  17593.18   142.857143 17736.037143           20.93      20.93         0.00   -2990.000000
     13    charge  17234.85   142.857143 17377.707143           22.21      22.93        -0.72  -15684.806286
     14    charge  17724.62   142.857143 17867.477143           21.60      21.60         0.

## (3). Further Cost Saving: "Price-Maker" vs. "Price-Taker"

The above analysis shows the

In a real market, supply isn't a smooth ramp; it’s a staircase of large power plants.

*   In HE 20, the discharge lowered the price by $ 0.17.

*   This 17-cent drop saved the utility $10,745 in a single hour. Why? Because that price drop applied to the entire 24,372 MW system load, not just our own 180 MW discharge. This vividly illustrates the benefit of utility-scale storage.
*   However, at peak hour, HE 19, the load reduction wasn't enough to cause a step-down in the clearing price.


Let's visualize the interplay by plotting the bid stack graphs before vs after.




In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Setup the 2-Panel Comparison
fig = make_subplots(
    rows=2, cols=1,  # Stacked vertically to see the "width" of the steps better
    subplot_titles=("HE 19: Peak Hour Market Impact", "HE 20: Post-Peak Market Impact"),
    vertical_spacing=0.15
)

for i, he in enumerate([19, 20], 1):
    stk = stacks_by_he[he].copy()
    load_before = hourly_load.loc[hourly_load["OPR_HR"] == he, "TOTAL_MW"].values[0]
    load_after = load_before - discharge_mw

    # Prices from your actual lookup function
    price_before = lookup_clearing_price(load_before, stk)
    price_after = lookup_clearing_price(load_after, stk)

    # Filter stack to the "Action Zone" (+/- 1000 MW around the load)
    # Use round(2) instead of round(0) to keep the REAL market precision
    stk_plot = stk[(stk['CUMULATIVE_MW'] > load_before - 1000) &
                   (stk['CUMULATIVE_MW'] < load_before + 500)]

    # --- DRAW THE SUPPLY STAIRCASE ---
    fig.add_trace(go.Scatter(
        x=stk_plot["CUMULATIVE_MW"], y=stk_plot["PRICE"],
        line=dict(color='#2980b9', shape='hv', width=3),
        name=f"Supply Stack HE {he}",
        hovertemplate="Price: $%{y:.2f}<br>Supply: %{x:,.0f} MW<extra></extra>"
    ), row=i, col=1)

    # --- DRAW THE BEFORE/AFTER INTERSECTIONS ---
    # Intersection 1: Baseline
    fig.add_trace(go.Scatter(
        x=[load_before, load_before, load_before - 500],
        y=[0, price_before, price_before],
        line=dict(color='#e74c3c', dash='dot'),
        name="Baseline Clearing", showlegend=False
    ), row=i, col=1)

    # Intersection 2: With Battery (The "New Price Level")
    fig.add_trace(go.Scatter(
        x=[load_after, load_after, load_after - 500],
        y=[0, price_after, price_after],
        line=dict(color='#27ae60', dash='dot'),
        name="New Clearing", showlegend=False
    ), row=i, col=1)

    # Add big markers at the exact clearing points
    fig.add_trace(go.Scatter(
        x=[load_before, load_after], y=[price_before, price_after],
        mode='markers+text',
        marker=dict(size=12, color=['#e74c3c', '#27ae60']),
        text=[f"${price_before:.2f}", f"${price_after:.2f}"],
        textposition="top center",
        showlegend=False
    ), row=i, col=1)

    # Zoom axes to see the "Drop"
    fig.update_xaxes(range=[load_before - 800, load_before + 300], row=i, col=1)
    fig.update_yaxes(range=[min(price_before, price_after) - 5, max(price_before, price_after) + 5], row=i, col=1)

fig.update_layout(height=800, width=900, title_text="Verification of Price Level Shift (HE 19 vs HE 20)", template="plotly_white")
fig.show()

The graph shows that load reduction via battery storage moved clearing market price from 36.85 USD to 36.68 USD during HE 20 (8PM), which could mean a very significant saving for the entire regional electricity system, but not during HE 19 (7PM).

**In HE20, the total cost saving = battery net benefit through price arbitrage + system net benefit through price supression**

## (4). System-Wide Effect: Total System Cost Analysis

What if the battery size is large enough to change the market clearing price even at HE 19, the peak hour? **When multiple large batteries all charge during the same cheap solar hours, they can collectively create a new price step — turning a previously flat region into a new clearing price.**

While the battery owner cares about their individual profit, the grid operator cares about the Total Area under the supply curve that everyone has to pay for.

In this section, we will analyze the impact on combined system cost savings of several large battery systems.


**(a). Exploring Minimal Load Threshold**:

To determine the optimal total size of these large battery storage systems, we will calculate the minimal load threshold required to reduce the price level during the peak hour.


In [ ]:
# ===========================================================================
# HE 19 Price Threshold Calculation
# ===========================================================================
print("=" * 65)
print(f"DIAGNOSTIC: Price Step Threshold for HE {PEAK_HE}")
print("=" * 65)

# Get the supply stack for HE 19
peak_stack = stacks_by_he[PEAK_HE].copy()
load_mw = peak_load_mw

# 1. Identify current clearing price
clearing_row = peak_stack[peak_stack["CUMULATIVE_MW"] >= load_mw].iloc[0]
clearing_price = clearing_row["PRICE"]

# 2. Find the start of the current price block
# This is the cumulative MW where the PREVIOUS (cheaper) price ended
prev_rows = peak_stack[peak_stack["PRICE"] < clearing_price]
flat_start_mw = float(prev_rows["CUMULATIVE_MW"].iloc[-1]) if not prev_rows.empty else 0.0

# 3. Calculate exact MW reduction needed to trigger a drop
# Any discharge > this value will push the net load into the cheaper price bucket
mw_threshold = load_mw - flat_start_mw

# 4. Calculate corresponding battery system size (MWh)
# Based on your discharge hours (5 hours) and efficiency (90%)
num_hours = len(DISCHARGE_HOURS)
required_mwh = (mw_threshold * num_hours) / ROUND_TRIP_EFF

print(f"Current HE {PEAK_HE} Load        : {load_mw:,.2f} MW")
print(f"Current Clearing Price     : ${clearing_price:.2f}/MWh")
print(f"Next Price Step below      : ${prev_rows.iloc[-1]['PRICE']:.2f}/MWh" if not prev_rows.empty else "N/A")
print("-" * 65)
print(f"MINIMUM DISCHARGE NEEDED   : {mw_threshold:,.2f} MW")
print(f"REQUIRED SYSTEM SIZE (MWh) : {required_mwh:,.2f} MWh")
print("-" * 65)

if discharge_mw >= mw_threshold:
    print(f"✅ Current {discharge_mw:.0f} MW battery is large enough.")
else:
    gap = mw_threshold - discharge_mw
    print(f"❌ Current {discharge_mw:.0f} MW battery is too small by {gap:,.2f} MW.")
    print(f"   The price will not move until you increase discharge to > {mw_threshold:,.2f} MW.")

DIAGNOSTIC: Price Step Threshold for HE 19
Current HE 19 Load        : 24,577.31 MW
Current Clearing Price     : $34.72/MWh
Next Price Step below      : $34.70/MWh
-----------------------------------------------------------------
MINIMUM DISCHARGE NEEDED   : 217.34 MW
REQUIRED SYSTEM SIZE (MWh) : 1,207.47 MWh
-----------------------------------------------------------------
❌ Current 180 MW battery is too small by 37.34 MW.
   The price will not move until you increase discharge to > 217.34 MW.


In [ ]:

# ===========================================================================
# Scan ALL hours: find which have a price-sensitive region
# near their load level (i.e. where a 250 MW battery would cross a step)
# ===========================================================================
print(f"\n{'=' * 65}")
print("DIAGNOSTIC: All hours — price reduction for 250 MW battery")
print("=" * 65)
print(f"  {'HE':>4}  {'Load MW':>10}  {'Base Price':>11}  {'Net Load':>10}  "
      f"{'New Price':>10}  {'ΔPrice':>8}  {'Step crosses?':>14}")
print("  " + "-" * 85)

scan_results = []
for he in range(1, 25):
    stk = build_supply_stack(bid_raw, TARGET_DATE, he)
    if stk is None:
        continue
    load_rows = hourly_load[hourly_load["OPR_HR"] == he]
    if load_rows.empty:
        continue
    load_mw    = float(load_rows["TOTAL_MW"].values[0])
    base_price = lookup_clearing_price(load_mw, stk)
    net_load   = load_mw - discharge_mw
    new_price  = lookup_clearing_price(net_load, stk)
    delta      = base_price - new_price
    crosses    = "✅ YES" if delta > 0 else "—"
    scan_results.append({
        "HE": he, "Load MW": load_mw, "Base Price": base_price,
        "Net Load": net_load, "New Price": new_price,
        "Delta": delta, "Crosses": crosses
    })
    print(f"  {he:>4}  {load_mw:>10,.1f}  {base_price:>11.2f}  {net_load:>10,.1f}  "
          f"{new_price:>10.2f}  {delta:>8.2f}  {crosses:>14}")

# Summary of hours with positive price reduction
positive = [r for r in scan_results if r["Delta"] > 0]
print(f"\n  Hours with positive price reduction: {len(positive)}")
for r in positive:
    print(f"    HE {r['HE']:>2}  load={r['Load MW']:,.0f} MW  "
          f"${r['Base Price']:.2f} → ${r['New Price']:.2f}  "
          f"(−${r['Delta']:.2f}/MWh)")


DIAGNOSTIC: All hours — price reduction for 250 MW battery
    HE     Load MW   Base Price    Net Load   New Price    ΔPrice   Step crosses?
  -------------------------------------------------------------------------------------
     1    20,679.1         2.50    20,499.1        2.40      0.10           ✅ YES
     2    19,929.3        13.44    19,749.3       13.44      0.00               —
     3    19,284.6        13.79    19,104.6       13.79      0.00               —
     4    18,947.5        12.67    18,767.5       12.63      0.04           ✅ YES
     5    18,905.5        11.11    18,725.5       11.11      0.00               —
     6    19,269.5        28.14    19,089.5       28.10      0.04           ✅ YES
     7    19,870.3         7.60    19,690.3        7.60      0.00               —
     8    20,070.8        15.14    19,890.8       15.14      0.00               —
     9    19,926.3         5.25    19,746.3        5.00      0.25           ✅ YES
    10    19,101.0         8.00 

The above analysis shows that our 250 MW storage already moves the clearing price for almost half of the hours, and it is close to move the peak-hour clearing price (only falls short by 37.34 MW).

For the purpose of analysis, let's imagine that California **adds three**  industrial battery storage / flexible load systems to the grid system.



---



Furthermore, since **charging increases the system cost (by pushing the price up)**, the timing of that charge is just as important as the discharge. We test three strategies to find the "Sweet Spot" for the grid.



**(b). Three Charging Strategies**:

1. **UNCOORDINATED** — all batteries charge/discharge on the same schedule (eg., Every battery operator looks at the same price forecast and says, "HE 10 is the cheapest!" They all charge at once.)

2. **STAGGERED**     — charge hours rotated round-robin across batteries (eg., A central operator (like CAISO) or a large utility says, "Fleet A charges at 10 AM, Fleet B at 11 AM.")

3. **OPTIMIZED**     — sequential greedy / game theory: each battery picks its cheapest charge window given prices already raised by earlier ones (eg., This mimics a real-world competitive market. Battery #1 picks the cheapest spot. Battery #2 looks at the new prices - now slightly higher because of Battery #1 - and picks the next best spot.)


In [ ]:
"""
CAISO Multi-Battery System-Wide Simulation
Three dispatch strategies for a fleet of N batteries:
  1. UNCOORDINATED — all batteries charge/discharge on the same schedule
  2. STAGGERED     — charge hours rotated round-robin across batteries
  3. OPTIMIZED     — sequential greedy: each battery picks its cheapest
                     charge window given prices already raised by earlier ones

Outputs:
  A — Per-hour detail for all three strategies (5× fleet)
  B — P&L comparison across strategies
  C — Charge-hour crowding-out table
  D — Fleet-size sweep 1–10 batteries (total benefit, suppression, arb)
  E — Best strategy by fleet size
"""


# ── CONFIG ─────────────────────────────────────────────────────────────────

BATTERY_CAPACITY_MWH = 1000
BATTERY_POWER_MW     = 250
ROUND_TRIP_EFF       = 0.90

CHARGE_HOURS    = list(range(9, 16))   # HE 9–15  (7 hrs, solar window)
DISCHARGE_HOURS = list(range(17, 22))  # HE 17–21 (5 hrs, evening peak)

N_BATTERIES = 4
SWEEP_RANGE = range(1, 11)

# ── DATA LOADING ────────────────────────────────────────────────────────────
def load_data():
    load_raw = pd.read_csv(LOAD_FILE)
    bid_raw  = pd.read_csv(BID_FILE)
    load_isotac = load_raw[
        (load_raw["TAC_AREA_NAME"] == "CA ISO-TAC") &
        (load_raw["OPR_DT"]        == TARGET_DATE)
    ].copy()
    load_isotac["OPR_HR"] = load_isotac["OPR_HR"].astype(int)
    hourly_load = (
        load_isotac
        .groupby("OPR_HR", as_index=False)
        .agg(TOTAL_MW=("MW", "sum"))
    ).sort_values("OPR_HR").reset_index(drop=True)
    return hourly_load, bid_raw

# ── BID STACK UTILITIES ─────────────────────────────────────────────────────
def build_supply_stack(bid_df, date, he):
    hour_start = f"{date}T{(he-1):02d}:00:00"
    bids = bid_df[
        (bid_df["RESOURCE_TYPE"]     == "GENERATOR") &
        (bid_df["SCH_BID_CURVETYPE"] == "BIDPRICE")  &
        (bid_df["STARTTIME"]         == hour_start)  &
        (bid_df["SCH_BID_XAXISDATA"].notna()) &
        (bid_df["SCH_BID_Y1AXISDATA"].notna())
    ].copy()
    bids["MW"]    = pd.to_numeric(bids["SCH_BID_XAXISDATA"],  errors="coerce")
    bids["PRICE"] = pd.to_numeric(bids["SCH_BID_Y1AXISDATA"], errors="coerce")
    bids = bids.dropna(subset=["MW","PRICE"])
    if bids.empty:
        return None
    bids["PRICE_BUCKET"] = bids["PRICE"].round(2)
    stk = pd.pivot_table(bids, values="MW", index="PRICE_BUCKET",
                         aggfunc="sum").reset_index()
    stk.columns = ["PRICE", "MW_AT_PRICE"]
    stk = stk.sort_values("PRICE").reset_index(drop=True)
    stk["CUMULATIVE_MW"] = stk["MW_AT_PRICE"].cumsum()
    return stk

def build_all_stacks(bid_raw, date, hours):
    return {he: stk
            for he in hours
            if (stk := build_supply_stack(bid_raw, date, he)) is not None}

def lookup_price(load_mw, stk):
    idx = np.searchsorted(stk["CUMULATIVE_MW"].values, load_mw, side="left")
    return float(stk.iloc[min(idx, len(stk)-1)]["PRICE"])

# ── SIMULATION ENGINE ────────────────────────────────────────────────────────
def simulate_scenario(hourly_load, stacks, mw_adj, label):
    """Apply MW adjustments, look up new prices, compute hourly savings."""
    rows = []
    for _, row in hourly_load.iterrows():
        he, load_mw = int(row["OPR_HR"]), float(row["TOTAL_MW"])
        stk = stacks.get(he)
        if stk is None:
            continue
        base_px  = lookup_price(load_mw, stk)
        adj      = mw_adj.get(he, 0.0)
        net_load = load_mw + adj
        new_px   = lookup_price(net_load, stk)
        rows.append({
            "OPR_HR"        : he,
            "ROLE"          : "discharge" if adj < 0 else ("charge" if adj > 0 else "none"),
            "TOTAL_MW"      : load_mw,
            "BATTERY_ADJ"   : adj,
            "NET_LOAD_MW"   : net_load,
            "CLEARING_PRICE": base_px,
            "NEW_PRICE"     : new_px,
            "PRICE_DELTA"   : base_px - new_px,
            "BASE_COST"     : base_px * load_mw,
            "NEW_COST"      : new_px  * net_load,
            "HOURLY_SAVING" : base_px * load_mw - new_px * net_load,
            "SCENARIO"      : label,
        })
    return pd.DataFrame(rows)

# ── THREE SCENARIOS ──────────────────────────────────────────────────────────

# 1. Simultaneous:
def scenario_uncoordinated(n, hourly_load, stacks):
    """All n batteries act together — maximum simultaneous MW impact."""
    fleet_cap = BATTERY_CAPACITY_MWH * n
    fleet_pwr = BATTERY_POWER_MW * n
    ch_energy = min(fleet_pwr * len(CHARGE_HOURS), fleet_cap)
    ch_mw     = ch_energy / len(CHARGE_HOURS)
    dis_mw    = min(fleet_pwr, ch_energy * ROUND_TRIP_EFF / len(DISCHARGE_HOURS))
    adj = {he: +ch_mw  for he in CHARGE_HOURS}
    adj.update({he: -dis_mw for he in DISCHARGE_HOURS})
    df = simulate_scenario(hourly_load, stacks, adj, f"Uncoordinated ({n}x)")
    return df, ch_mw, dis_mw


# 2. Smoothing:
def scenario_staggered(n, hourly_load, stacks):
    """
    Rotate charge hours round-robin: battery i charges hours where idx%n==i.
    Keeps per-hour charge MW ≈ BATTERY_POWER_MW instead of n×BATTERY_POWER_MW.
    All batteries still discharge together (peak suppression unchanged).
    """
    per_batt = [CHARGE_HOURS[i::n] for i in range(n)]
    charge_adj, total_stored = {}, 0.0
    for hours_i in per_batt:
        if not hours_i:
            continue
        energy_i = min(BATTERY_POWER_MW * len(hours_i), BATTERY_CAPACITY_MWH)
        mw_i     = energy_i / len(hours_i)
        total_stored += energy_i * ROUND_TRIP_EFF
        for he in hours_i:
            charge_adj[he] = charge_adj.get(he, 0.0) + mw_i
    dis_mw = min(BATTERY_POWER_MW * n, total_stored / len(DISCHARGE_HOURS))
    adj = dict(charge_adj)
    adj.update({he: -dis_mw for he in DISCHARGE_HOURS})
    return simulate_scenario(hourly_load, stacks, adj, f"Staggered ({n}x)")

# 3. Sequential greedy:
def scenario_optimized(n, hourly_load, stacks):
    """
    Sequential greedy dispatch.
    Battery 1 picks the cheapest charge hours → updates net loads.
    Battery 2 sees the now-elevated prices and picks the next-cheapest window.
    … and so on. Accurately models crowding-out among fleet members.
    """
    net_loads = hourly_load.set_index("OPR_HR")["TOTAL_MW"].to_dict()
    combined_adj = {}
    candidates = [he for he in CHARGE_HOURS if he in stacks]

    for b in range(n):
        # Price each candidate hour at current net load
        prices = {he: lookup_price(net_loads[he], stacks[he]) for he in candidates}
        sorted_hrs = sorted(candidates, key=lambda h: prices[h])

        # Greedily fill this battery's capacity at cheapest hours first
        energy_rem, chosen = BATTERY_CAPACITY_MWH, {}
        for he in sorted_hrs:
            if energy_rem <= 0:
                break
            mw = min(BATTERY_POWER_MW, energy_rem)
            chosen[he], energy_rem = mw, energy_rem - mw

        # Commit charge, update net loads
        for he, mw in chosen.items():
            net_loads[he]     = net_loads.get(he, 0) + mw
            combined_adj[he]  = combined_adj.get(he, 0) + mw

        # Spread discharge evenly across peak hours
        dis_mw = min(BATTERY_POWER_MW,
                     sum(chosen.values()) * ROUND_TRIP_EFF / len(DISCHARGE_HOURS))
        for he in DISCHARGE_HOURS:
            net_loads[he]     = net_loads.get(he, 0) - dis_mw
            combined_adj[he]  = combined_adj.get(he, 0) - dis_mw

    adj = {he: v for he, v in combined_adj.items() if v != 0}
    return simulate_scenario(hourly_load, stacks, adj, f"Optimized ({n}x)")

# ── P&L SUMMARY ─────────────────────────────────────────────────────────────
def pnl_summary(df):
    """
    Calculates the financial impact of the battery fleet.
    Total Social Benefit = Sum of hourly market expenditure changes.
    """
    active  = df[df["ROLE"] != "none"]
    by_role = (
        active.groupby("ROLE")
        .agg(HOURS=("OPR_HR","count"),
             TOTAL_SAVING=("HOURLY_SAVING","sum"),
             AVG_BASE_PX=("CLEARING_PRICE","mean"),
             AVG_NEW_PX=("NEW_PRICE","mean"))
        .reset_index()
    )

    # 1. Total Social Benefit (The 'Economic Surplus' created)
    # Sum of (Base Expenditure - New Expenditure) across all hours.
    total_social_benefit = df["HOURLY_SAVING"].sum()

    # 2. Battery Private Profit (Arbitrage)
    # Calculated using NEW prices (post-suppression)
    dis_rows = active[active["ROLE"]=="discharge"]
    ch_rows  = active[active["ROLE"]=="charge"]

    discharge_rev = (dis_rows["NEW_PRICE"] * dis_rows["BATTERY_ADJ"].abs()).sum()
    charge_cost   = (ch_rows["NEW_PRICE"] * ch_rows["BATTERY_ADJ"]).sum() / ROUND_TRIP_EFF

    battery_profit = discharge_rev - charge_cost

    # 3. Consumer Savings (Pure suppression benefit)
    consumer_savings = total_social_benefit - battery_profit

    avg_dis = dis_rows["NEW_PRICE"].mean() if not dis_rows.empty else 0.0
    avg_ch  = ch_rows["NEW_PRICE"].mean() if not ch_rows.empty else 0.0

    return {
        "SCENARIO"        : df["SCENARIO"].iloc[0],
        "TOTAL_BENEFIT"   : total_social_benefit,
        "ARB_REVENUE"     : battery_profit,
        "NET_SUPPRESSION" : consumer_savings,
        "AVG_DISCHARGE_PX": avg_dis,
        "AVG_CHARGE_PX"   : avg_ch,
        "ARB_SPREAD"      : avg_dis - avg_ch,
        "by_role"         : by_role,
    }

# ── FLEET SWEEP ──────────────────────────────────────────────────────────────
def fleet_sweep(hourly_load, stacks):
    rows = []
    for n in SWEEP_RANGE:
        df_unc, *_ = scenario_uncoordinated(n, hourly_load, stacks)
        df_stg     = scenario_staggered(n, hourly_load, stacks)
        df_opt     = scenario_optimized(n, hourly_load, stacks)
        for df in [df_unc, df_stg, df_opt]:
            s   = pnl_summary(df)
            scn = s["SCENARIO"].split("(")[0].strip()
            rows.append({"N_BATTERIES": n, "SCENARIO": scn,
                         "NET_SUPPRESSION": round(s["NET_SUPPRESSION"], 2),
                         "ARB_REVENUE"    : round(s["ARB_REVENUE"],     2),
                         "TOTAL_BENEFIT"  : round(s["TOTAL_BENEFIT"],   2)})
    return pd.DataFrame(rows)

# ── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    print("Loading data …")
    hourly_load, bid_raw = load_data()
    all_hrs = sorted(set(CHARGE_HOURS + DISCHARGE_HOURS + list(hourly_load["OPR_HR"])))
    print(f"Building supply stacks for {len(all_hrs)} hours …")
    stacks = build_all_stacks(bid_raw, TARGET_DATE, all_hrs)
    print(f"  Stacks ready: {len(stacks)} hours\n")

    # ── A: Per-hour detail ──────────────────────────────────────────────────
    print("=" * 72)
    print(f"PART A: Hour-by-Hour Results — {N_BATTERIES}× fleet")
    print("=" * 72)
    df_unc, ch_mw, dis_mw = scenario_uncoordinated(N_BATTERIES, hourly_load, stacks)
    df_stg = scenario_staggered(N_BATTERIES, hourly_load, stacks)
    df_opt = scenario_optimized(N_BATTERIES, hourly_load, stacks)
    print(f"\n  Fleet: {N_BATTERIES}×{BATTERY_POWER_MW}MW/{BATTERY_CAPACITY_MWH}MWh  "
          f"RTE={ROUND_TRIP_EFF:.0%}  "
          f"| uncoord charge {ch_mw:.0f}MW/hr  discharge {dis_mw:.0f}MW/hr\n")
    cols = ["OPR_HR","ROLE","TOTAL_MW","BATTERY_ADJ","NET_LOAD_MW",
            "CLEARING_PRICE","NEW_PRICE","PRICE_DELTA","HOURLY_SAVING"]
    for label, df in [("UNCOORDINATED", df_unc),
                      ("STAGGERED",     df_stg),
                      ("OPTIMIZED",     df_opt)]:
        print(f"  ── {label} ──")
        print(df[df["ROLE"]!="none"][cols].to_string(index=False))
        print()

    # ── B: P&L comparison ──────────────────────────────────────────────────
    print("=" * 72)
    print(f"PART B: P&L Comparison — {N_BATTERIES}× fleet  (all $ per day)")
    print("=" * 72)
    summaries = [pnl_summary(df) for df in [df_unc, df_stg, df_opt]]
    comp = pd.DataFrame([{
        "SCENARIO"       : s["SCENARIO"],
        "NET_SUPPRESSION": round(s["NET_SUPPRESSION"], 2),
        "ARB_REVENUE"    : round(s["ARB_REVENUE"],     2),
        "TOTAL_BENEFIT"  : round(s["TOTAL_BENEFIT"],   2),
        "AVG_DIS_PX"     : round(s["AVG_DISCHARGE_PX"],2),
        "AVG_CHG_PX"     : round(s["AVG_CHARGE_PX"],   2),
        "ARB_SPREAD"     : round(s["ARB_SPREAD"],       2),
    } for s in summaries])
    print(comp.to_string(index=False))
    print("\n  Role breakdown per scenario:")
    for s in summaries:
        print(f"\n    {s['SCENARIO']}")
        print(s["by_role"].to_string(index=False))

    # ── C: Crowding-out at charge hours ────────────────────────────────────
    print("\n" + "=" * 72)
    print("PART C: Charge-Hour Crowding-Out — MW injected & new price per strategy")
    print("=" * 72)
    base = df_unc[df_unc["ROLE"]=="charge"][["OPR_HR","CLEARING_PRICE","TOTAL_MW"]].copy()
    for df, tag in [(df_unc,"UNC"),(df_stg,"STG"),(df_opt,"OPT")]:
        sub = df[df["ROLE"]=="charge"][["OPR_HR","BATTERY_ADJ","NEW_PRICE","HOURLY_SAVING"]]
        sub = sub.rename(columns={"BATTERY_ADJ":f"ADJ_{tag}",
                                   "NEW_PRICE":f"PX_{tag}",
                                   "HOURLY_SAVING":f"SAV_{tag}"})
        base = base.merge(sub, on="OPR_HR", how="left")
    print(base.to_string(index=False))

    # ── D: Fleet-size sweep ────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("PART D: Fleet-Size Sweep  (1 – 10 batteries)")
    print("=" * 72)
    sweep_df = fleet_sweep(hourly_load, stacks)
    pivot = sweep_df.pivot_table(
        index="N_BATTERIES", columns="SCENARIO",
        values=["TOTAL_BENEFIT","NET_SUPPRESSION","ARB_REVENUE"]
    )
    pivot.columns = [f"{m}_{s}" for m,s in pivot.columns]
    pivot = pivot.reset_index()

    for title, tag in [("Total daily benefit (USD)", "TOTAL_BENEFIT"),
                       ("Net price suppression (USD)", "NET_SUPPRESSION"),
                       ("Arbitrage revenue (USD)", "ARB_REVENUE")]:
        print(f"\n  {title}:")
        cols_t = ["N_BATTERIES"] + [c for c in pivot.columns if tag in c]
        print(pivot[cols_t].to_string(index=False))

    # Marginal battery analysis — uncoordinated
    print("\n  Marginal benefit of the n-th battery (UNCOORDINATED):")
    unc_s = sweep_df[sweep_df["SCENARIO"]=="Uncoordinated"].sort_values("N_BATTERIES").copy()
    unc_s["MARGINAL"] = unc_s["TOTAL_BENEFIT"].diff()
    print(unc_s[["N_BATTERIES","TOTAL_BENEFIT","MARGINAL"]].to_string(index=False))
    neg = unc_s[unc_s["MARGINAL"] < 0]
    if not neg.empty:
        print(f"\n  ⚠️  Marginal benefit turns NEGATIVE at battery #{int(neg.iloc[0]['N_BATTERIES'])} "
              f"(uncoordinated) — crowding-out exceeds new arbitrage gains.")
    else:
        print(f"\n  ✅ Marginal benefit stays positive across full sweep.")

    # ── E: Best strategy per fleet size ───────────────────────────────────
    print("\n" + "=" * 72)
    print("PART E: Best Strategy by Fleet Size")
    print("=" * 72)
    winner = (
        sweep_df.sort_values("TOTAL_BENEFIT", ascending=False)
        .groupby("N_BATTERIES").first().reset_index()
        [["N_BATTERIES","SCENARIO","TOTAL_BENEFIT","NET_SUPPRESSION","ARB_REVENUE"]]
    )
    print(winner.to_string(index=False))
    return sweep_df, df_unc, df_stg, df_opt, hourly_load, stacks

if __name__ == "__main__":
    sweep_df, df_unc, df_stg, df_opt, hourly_load, stacks = main()

Loading data …
Building supply stacks for 24 hours …
  Stacks ready: 24 hours

PART A: Hour-by-Hour Results — 4× fleet

  Fleet: 4×250MW/1000MWh  RTE=90%  | uncoord charge 571MW/hr  discharge 720MW/hr

  ── UNCOORDINATED ──
 OPR_HR      ROLE  TOTAL_MW  BATTERY_ADJ  NET_LOAD_MW  CLEARING_PRICE  NEW_PRICE  PRICE_DELTA  HOURLY_SAVING
      9    charge  19926.29   571.428571 20497.718571            5.25       6.91        -1.66  -37026.212829
     10    charge  19101.01   571.428571 19672.438571            8.00       8.16        -0.16   -7719.018743
     11    charge  18369.73   571.428571 18941.158571            6.39       6.87        -0.48  -12743.184686
     12    charge  17593.18   571.428571 18164.608571            7.33       7.42        -0.09   -5823.386200
     13    charge  17234.85   571.428571 17806.278571           13.62      14.71        -1.09  -27191.700786
     14    charge  17724.62   571.428571 18296.048571            7.89       7.89         0.00   -4508.571429
     15    ch

**Analysis:**

**#1. The "Death by Success" Threshold (Part D & E)**

The most striking result is in Part D. Look at the Marginal Benefit for the Uncoordinated scenario:

*  Battery 3: Adding this battery increased total benefit by $30,313.

*   Battery 4: Adding this one caused the marginal benefit to drop to -$286.

*   Battery 6+: The optimized strategy's total benefit turns massively negative (-$46k to -$105k).

The Verdict: the electricity grid has a "carrying capacity" for this specific supply stack. Once we hit 4 batteries (1,000 MW total power), the batteries start "**crowding each other out**." They raise the price of charging so much that they destroy the very value they were trying to create. **4 batteries is the optimal size.**



---


**#2. Optimized vs. Staggered: The "Slow and Steady" Winner**

In Part E, look at what happens when the fleet grows to **8–10 batteries**:


*   **Winner: Staggered (gradual and coordinated)**.


Why? When the fleet is huge, the "Optimized" and "Uncoordinated" strategies are too aggressive. They try to shove too much energy into a few hours, causing massive price spikes (look at PX_OPT in HE 9 hitting 7.00 from a base of 5.25 USD).

**The Lesson: In a saturated market, a central "Staggered" coordinator is the only way to keep the system from collapsing into negative benefits.**


---



**#3. Why Uncoordinated "Won" at Battery #3**

I was initially surprised that Uncoordinated beat Optimized for a 3-battery fleet. However, sometimes "dumb" luck is better than "greedy" math. By spreading the charge across 7 hours (Uncoordinated) instead of 4 hours (Optimized), the "dumb" strategy hit fewer "price cliffs" in the supply stack.

**My Takeaway: Spreading load (lower MW over more hours) is almost always better for the grid than concentrated load (high MW over fewer hours).**

## Conclusion: The "Price-Maker" Paradox
The initial hypothesis—that a large battery fleet could significantly lower peak prices through Price Suppression—has been validated. However, there is a critical caveat. While the 180 MW "Price-Maker" effect successfully shaved the evening peaks in HE 18–20, the project revealed a hidden "Crowding-Out" effect during the charging window. The simulation proves that battery value is not linear; it is highly sensitive to the Supply Stack's "Price Cliffs."

**Project Executive Summary**

1. The Mechanism: We modeled a 250 MW / 1,000 MWh battery fleet against real CAISO supply-stack data, treating the batteries as "Price-Makers" rather than "Price-Takers."

2. The Peak Impact: In high-demand hours, the battery successfully "jumped" the market down to cheaper generator biddings, creating massive System-Wide Savings that far exceeded the battery's private arbitrage profits.

3. The Charging Struggle: We discovered that how you refuel matters as much as how you discharge. "Uncoordinated" charging (all batteries hitting the same hour) creates "Secondary Peaks" that can accidentally raise costs for all grid users.

4. The Optimal Strategy: Staggered Coordination emerged as the most resilient strategy for a large-scale future. While "Optimized" (Greedy) bidding maximizes private profit in the short term, it leads to "Diminishing Marginal Returns" once the fleet exceeds 1,000 MW.

Storage is the ultimate tool for grid stability, but only if managed as a coordinated fleet. Without coordination, the "Green Charging" window becomes a new bottleneck, and the batteries begin to cannibalize their own economic value.



---

## Citation:

1. CAISO OASIS, Nov 30 2025 Data: https://oasis.caiso.com/mrioasis/logon.do?reason=application.baseAction.noSession#
2. CPUC, 2023: https://www.cpuc.ca.gov/-/media/cpuc-website/divisions/energy-division/documents/integrated-resource-plan-and-long-term-procurement-plan-irp-ltpp/2023-irp-cycle-events-and-materials/2022-2023psp_decision_2pager_final.pdf
3. CEC, 2024: https://www.energy.ca.gov/programs-and-topics/topics/load-flexibility
4. Claude AI (assisted with coding).
